In [2]:
!pip install pandas numpy scikit-surprise

  Using cached scikit_surprise-1.1.4-cp312-cp312-win_amd64.whl
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 299.6 kB/s eta 0:00:31
   -- ------------------------------------- 0.5/9.7 MB 299.6 kB/s eta 0:00:31
   -- ------------------------------------- 0.5/9.7 MB 299.6 kB/s eta 0:00:31
   -- ------------------------------------- 0.5/9.7 MB 299.6 kB/s eta 0:00:31
   -- ------------------------------------- 0.5/9.7 MB 299.6 kB/s eta 0:0

In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
import pickle
import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:

interactions = pd.read_csv("data/interactions.csv")


cf_data = interactions[['player_id', 'item_id', 'rating']]

print(f"Dataset shape: {cf_data.shape}")
print(cf_data.head())

Dataset shape: (688, 3)
   player_id  item_id  rating
0          1        2       3
1          1       48       2
2          1       18       3
3          1       16       5
4          1       15       2


In [ ]:

reader = Reader(rating_scale=(1, 5))


data = Dataset.load_from_df(cf_data[['player_id', 'item_id', 'rating']], reader)

print("✅ Data converted to Surprise format!")

✅ Data converted to Surprise format!


In [ ]:

trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

print(f"Training samples: {trainset.n_ratings}")
print(f"Test samples: {len(testset)}")

Training samples: 550
Test samples: 138


In [ ]:

model = SVD(n_factors=50, n_epochs=20, random_state=42)


model.fit(trainset)

print("✅ Model training complete!")

✅ Model training complete!


In [ ]:

predictions = model.test(testset)


rmse_score = accuracy.rmse(predictions)
mae_score = accuracy.mae(predictions)

print(f"\nSummary:\nRMSE: {rmse_score:.4f}\nMAE: {mae_score:.4f}")
if rmse_score < 1.0:
    print("Result: Good model accuracy!")
else:
    print("Result: Needs improvement.")

RMSE: 1.1673
MAE:  1.0330

Summary:
RMSE: 1.1673
MAE: 1.0330
Result: Needs improvement.


In [ ]:
def get_cf_recommendations(player_id, n=5):
    
    rated_items = interactions[interactions['player_id'] == player_id]['item_id'].tolist()
    
    
    items_df = pd.read_csv("data/items.csv")
    all_item_ids = items_df['item_id'].tolist()
    
    
    unseen_items = [item for item in all_item_ids if item not in rated_items]
    
    
    preds = []
    for item_id in unseen_items:
        est_rating = model.predict(player_id, item_id).est
        preds.append((item_id, est_rating))
    
    
    preds.sort(key=lambda x: x[1], reverse=True)
    top_preds = preds[:n]
    
    
    results = []
    for item_id, score in top_preds:
        item_name = items_df[items_df['item_id'] == item_id]['item_name'].values[0]
        results.append({"item_id": item_id, "item_name": item_name, "predicted_score": round(score, 2)})
        
    return results


print(f"Top 5 recommendations for Player 1: \n{get_cf_recommendations(1)}")

Top 5 recommendations for Player 1: 
[{'item_id': 25, 'item_name': 'Golden Warrior', 'predicted_score': 4.16}, {'item_id': 31, 'item_name': 'Zombie Survivor', 'predicted_score': 4.09}, {'item_id': 39, 'item_name': 'Vehicle Lava Buggy', 'predicted_score': 4.02}, {'item_id': 17, 'item_name': 'Rifle QBZ', 'predicted_score': 3.98}, {'item_id': 8, 'item_name': 'Shotgun DBS', 'predicted_score': 3.95}]
